# RQ3 — Prompt Size vs Latency & Quality Benchmark

**What this measures:**
- How does latency scale with prompt size (number of tokens)?
- Does a larger prompt degrade visualization quality?

**Method:**
- 5 prompt sizes from ~200 to ~4000 tokens (XS to XL)
- 5 runs per size = 25 total experiments
- SA5 architecture (router + single-shot extract, gpt-4o-mini)
- Same 8-check quality validator as architecture benchmark
- ~5-10 minutes total runtime

## Cell 1 — Setup

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Setup
# Prompt Size vs Latency + Quality Benchmark
# ═══════════════════════════════════════════════════════════════════
import sys, os, time, json, copy, csv, traceback
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

CANDIDATES = [
    "/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch",
    os.path.abspath(".."), os.getcwd(),
]
PROJECT_ROOT = next(
    (p for p in CANDIDATES if os.path.isfile(os.path.join(p, "generate_visualization.py"))), None
)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from retrieve_data import retrieve_data
from generate_visualization import Agent
from init_phoenix import init_phoenix
from visualization_from_template import generate_from_template

import phoenix as px
if not hasattr(px, "_thesis_initialised"):
    client, tool_client, tracer = init_phoenix("rq3-prompt-size")
    px._thesis_initialised = True
    px._client = client
    px._tool = tool_client
    px._tracer = tracer
else:
    client = px._client
    tool_client = px._tool
    tracer = px._tracer

MD_TABLE_FULL = retrieve_data(None, type="test")
print(f"Full data table: {len(MD_TABLE_FULL)} chars")

os.makedirs("rq3_results", exist_ok=True)
os.makedirs("rq3_results/renders", exist_ok=True)


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: rq3-prompt-size
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Full data table: 3405 chars


## Cell 2 — Create 5 data slices

In [2]:
conda install tiktoken

Solving environment: - 
The environment is inconsistent, please check the package plan carefully
The following packages are causing the inconsistency:

  - conda-forge/osx-64::pytorch==2.4.1=cpu_generic_py311h490ac94_0
  - conda-forge/osx-64::pandas==2.3.3=py311hca9a5ca_2
  - conda-forge/noarch::transformers==4.48.3=pyhd8ed1ab_0
  - defaults/osx-64::pyarrow==19.0.0=py311h6d0c2b6_1
  - conda-forge/osx-64::libtorch==2.4.1=cpu_generic_h70335e3_0
  - conda-forge/noarch::sentence-transformers==5.2.3=pyhd8ed1ab_0
  - conda-forge/osx-64::scikit-learn==1.8.0=np2py311heeb2b4f_1
  - conda-forge/noarch::datasets==4.0.0=pyhcf101f3_0
done


==> WARNING: A newer version of conda exists. <==
  current version: 23.7.4
  latest version: 25.7.0

Please update conda by running

    $ conda update -n base -c defaults conda

Or to minimize the number of packages updated during conda update use

     conda install conda=25.7.0



## Package Plan ##

  environment location: /Users/srujanakadambari/anaconda3/

In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Create 5 data slices with different prompt sizes
# ═══════════════════════════════════════════════════════════════════
import tiktoken

def count_tokens(text):
    try:
        enc = tiktoken.encoding_for_model("04-mini")
    except:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))

# Parse the full table
lines = MD_TABLE_FULL.strip().split("\n")
# find header + separator rows
header_idx = 0
for i, line in enumerate(lines):
    if line.strip().startswith("|") and "---" not in line:
        header_idx = i
        break

header = lines[header_idx]
sep = lines[header_idx+1] if header_idx+1 < len(lines) and "---" in lines[header_idx+1] else "|---|---|---|"
data_rows = [l for l in lines[header_idx+2:] if l.strip().startswith("|")]

print(f"Total data rows available: {len(data_rows)}")

# Create 5 sizes
sizes = {
    "XS_1year":   [r for r in data_rows if "2024" in r[:30]],           # ~12 rows
    "S_2years":   [r for r in data_rows if any(y in r[:30] for y in ["2023","2024"])],
    "M_3years":   [r for r in data_rows if any(y in r[:30] for y in ["2022","2023","2024"])],
    "L_4years":   data_rows,                                              # full
    "XL_padded":  data_rows,                                              # full + padding
}

SLICES = {}
for name, rows in sizes.items():
    table = "\n".join([header, sep] + rows)
    if name == "XL_padded":
        # Add padding context to make it ~4000 tokens
        padding = ("Additional business context: The JVA segment (Justizvollzugsanstalten) represents "
                   "correctional facility contracts. Revenue is measured in EUR. Monthly data from "
                   "January 2021 through December 2024. Teckentrup is a German door manufacturer. "
                   "Segment SD/CO stands for Sales Division Contract Orders. " * 15)
        table = padding + "\n\n" + table
    
    SLICES[name] = {
        "data": table,
        "char_count": len(table),
        "token_count": count_tokens(table),
        "n_rows": len(rows),
    }
    print(f"  {name:<12} {len(rows):>3} rows  {SLICES[name]['char_count']:>6} chars  {SLICES[name]['token_count']:>5} tokens")

QUESTION = (
    "Wieviel Umsatz hatte Teckentrup in den Jahren 2021 bis 2024 im Segment JVA? "
    "Provide a detailed analysis and a comprehensive visualization."
)


Total data rows available: 48
  XS_1year      12 rows     943 chars    245 tokens
  S_2years      24 rows    1759 chars    450 tokens
  M_3years      36 rows    2575 chars    656 tokens
  L_4years      48 rows    3391 chars    862 tokens
  XL_padded     48 rows    7833 chars   1823 tokens


## Cell 3 — Quality validator

In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Quality validator (same 8-check as other benchmarks)
# ═══════════════════════════════════════════════════════════════════
VALID_CT = {"line","bar","pie","scatter","column","stackedbar"}

def validate(cfg, render=True, tag="?", run=0):
    r = {"schema_complete":False,"valid_charttype":False,"has_data":False,
         "has_title":False,"has_axes":False,"renderable":False,
         "has_annotations":False,"data_consistency":False,
         "quality_score":0,"passed":False,"error":None,
         "n_data_groups":0,"n_annotations":0}
    
    if cfg is None: r["error"]="null"; return r
    if hasattr(cfg, "model_dump"): cfg = cfg.model_dump()
    if not isinstance(cfg, dict): r["error"]="not dict"; return r
    
    req = ["titlename","charttype","xlabel","ylabel","data"]
    r["schema_complete"] = all(cfg.get(f) for f in req)
    
    ct = cfg.get("charttype","")
    if hasattr(ct,"value"): ct=ct.value
    r["valid_charttype"] = str(ct).lower().strip() in VALID_CT
    
    data = cfg.get("data") or []
    r["has_data"] = len(data) > 0
    r["n_data_groups"] = len(data)
    r["has_title"] = bool(str(cfg.get("titlename","")).strip())
    r["has_axes"] = bool(cfg.get("xlabel")) and bool(cfg.get("ylabel"))
    
    consistent = True
    for g in data:
        d = g if isinstance(g,dict) else (g.model_dump() if hasattr(g,"model_dump") else {})
        xd, yd = d.get("x_data",[]) or [], d.get("y_data",[]) or []
        if not xd or not yd or len(xd) != len(yd): consistent=False; break
    r["data_consistency"] = consistent
    
    annos = cfg.get("annotations") or []
    r["has_annotations"] = len(annos) > 0
    r["n_annotations"] = len(annos)
    
    if render and r["has_data"] and r["valid_charttype"]:
        try:
            plt.close("all")
            generate_from_template(copy.deepcopy(cfg))
            fn = f"rq3_results/renders/{tag}_run{run:02d}.png"
            plt.savefig(fn, dpi=100, bbox_inches="tight")
            plt.close("all")
            r["renderable"] = True
        except Exception as e:
            r["error"] = f"render: {str(e)[:60]}"
            plt.close("all")
    
    checks = ["schema_complete","valid_charttype","has_data","has_title",
              "has_axes","renderable","has_annotations","data_consistency"]
    r["quality_score"] = sum(r[c] for c in checks)
    r["passed"] = r["quality_score"] >= 5
    return r

print("Validator ready")


Validator ready


## Cell 4 — Run benchmark (25 total calls)

In [6]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Run benchmark: 5 sizes × 5 runs each on SA5 architecture
# (SA5 = router + single-shot extract with o4o-mini)
# ═══════════════════════════════════════════════════════════════════
from generate_visualization import Agent

N_RUNS = 5  # 5 runs per size for time budget
agent = Agent(client, tool_client, tracer, "o4-mini")

CSV_FILE = "rq3_results/prompt_size_benchmark.csv"
fields = ["size","n_rows","char_count","token_count","run","latency_s",
          "quality_score","passed","schema_complete","valid_charttype","has_data",
          "renderable","has_annotations","data_consistency","n_data_groups","n_annotations","error"]

with open(CSV_FILE,"w",newline="") as f:
    csv.DictWriter(f, fieldnames=fields).writeheader()

print(f"Running {len(SLICES)} sizes × {N_RUNS} runs = {len(SLICES)*N_RUNS} calls\n")

for size_name, slc in SLICES.items():
    print(f"{'='*65}")
    print(f"  {size_name}  ({slc['token_count']} tokens, {slc['n_rows']} rows)")
    print(f"{'='*65}")
    
    for i in range(N_RUNS):
        print(f"  Run {i+1}/{N_RUNS} ... ", end="", flush=True)
        t0 = time.perf_counter()
        try:
            raw = agent.extract_chart_config(slc["data"], analysis=QUESTION)
            lat = round(time.perf_counter()-t0, 3)
            if hasattr(raw,"model_dump"): raw = raw.model_dump()
            if isinstance(raw,dict) and "content" in raw:
                c = raw["content"]
                raw = json.loads(c) if isinstance(c,str) else c
            qr = validate(raw, render=True, tag=size_name, run=i+1)
            status = f"PASS {qr['quality_score']}/8" if qr["passed"] else f"FAIL {qr.get('error','?')[:40]}"
            print(f"{lat:6.2f}s  {status}")
        except Exception as e:
            lat = round(time.perf_counter()-t0, 3)
            qr = {k:False for k in ["schema_complete","valid_charttype","has_data","renderable",
                                     "has_annotations","data_consistency","passed"]}
            qr.update({"quality_score":0,"n_data_groups":0,"n_annotations":0,
                       "error":str(e)[:80]})
            print(f"{lat:6.2f}s  CRASH")
        
        row = {"size":size_name,"n_rows":slc["n_rows"],"char_count":slc["char_count"],
               "token_count":slc["token_count"],"run":i+1,"latency_s":lat,
               **{k:qr.get(k) for k in fields[6:]}}
        with open(CSV_FILE,"a",newline="") as f:
            csv.DictWriter(f, fieldnames=fields).writerow(row)

print(f"\nSaved: {CSV_FILE}")


Running 5 sizes × 5 runs = 25 calls

  XS_1year  (245 tokens, 12 rows)
  Run 1/5 ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 16.20s  PASS 8/8
  Run 2/5 ...  13.01s  PASS 8/8
  Run 3/5 ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 16.85s  PASS 7/8
  Run 4/5 ... 

...

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 20.49s  PASS 7/8
  Run 5/5 ... 

/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


 19.67s  PASS 7/8

Saved: rq3_results/prompt_size_benchmark.csv


/Users/srujanakadambari/Desktop/ffresh thesis/data-to-visual/data-to-visual-nicos-branch/visualization_from_template.py:398: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ═══════════════════════════════════════════════════════════════════
# CLEANUP — drop the incomplete XL_padded runs so Cell 5 works cleanly
# ═══════════════════════════════════════════════════════════════════
import pandas as pd

df = pd.read_csv("rq3_results/prompt_size_benchmark.csv")
print(f"Before cleanup: {len(df)} rows")
print(df.groupby("size").size())

# keep only completed sizes (drop XL if it has fewer than 5 runs)
counts = df.groupby("size").size()
complete_sizes = counts[counts >= 5].index.tolist()
df_clean = df[df["size"].isin(complete_sizes)]

print(f"\nAfter cleanup: {len(df_clean)} rows")
print(df_clean.groupby("size").size())

df_clean.to_csv("rq3_results/prompt_size_benchmark.csv", index=False)
print("\nSaved cleaned CSV")

Before cleanup: 25 rows
size
L_4years     5
M_3years     5
S_2years     5
XL_padded    5
XS_1year     5
dtype: int64

After cleanup: 25 rows
size
L_4years     5
M_3years     5
S_2years     5
XL_padded    5
XS_1year     5
dtype: int64

Saved cleaned CSV


## Cell 5 — Analysis + figures

In [8]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Analysis: latency vs tokens + quality per size
# ═══════════════════════════════════════════════════════════════════
import matplotlib.patches as mpatches

df = pd.read_csv(CSV_FILE)
df["latency_s"] = pd.to_numeric(df["latency_s"], errors="coerce")
df["quality_score"] = pd.to_numeric(df["quality_score"], errors="coerce")
df["passed"] = df["passed"].astype(str).isin(["True","1","1.0"])

summary = df.groupby("size").agg(
    tokens=("token_count","first"),
    rows=("n_rows","first"),
    lat_mean=("latency_s","mean"),
    lat_std=("latency_s","std"),
    qual_mean=("quality_score","mean"),
    pass_rate=("passed","mean")
).reset_index()
summary["pass_rate"] = summary["pass_rate"] * 100

# order by size
ORDER = ["XS_1year","S_2years","M_3years","L_4years"]
summary = summary.set_index("size").reindex(ORDER).reset_index()
print(summary.to_string())

# ── FIG 1: Latency vs prompt size (with correlation) ────────────────
fig, ax = plt.subplots(figsize=(11,6))
x = summary["tokens"].values
y = summary["lat_mean"].values
yerr = summary["lat_std"].values

# scatter with error bars
ax.errorbar(x, y, yerr=yerr, fmt="o", markersize=12, capsize=8,
            color="#2E75B6", ecolor="#2E75B6", linewidth=2, label="Mean ± std")

# linear fit
if len(x) >= 2:
    slope, intercept = np.polyfit(x, y, 1)
    xline = np.linspace(x.min(), x.max(), 100)
    ax.plot(xline, slope*xline+intercept, "--", color="#C0392B",
            label=f"Linear fit: y = {slope:.4f}x + {intercept:.2f}")
    r = np.corrcoef(x, y)[0,1]
    ax.text(0.05, 0.95, f"Pearson r = {r:.3f}\nSlope = {slope*1000:.2f}s per 1k tokens",
            transform=ax.transAxes, fontsize=11, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9))

# label points
for i, row in summary.iterrows():
    ax.annotate(f"{row['size']}\n({int(row['tokens'])}tok)",
                (row["tokens"], row["lat_mean"]),
                textcoords="offset points", xytext=(12,-5), fontsize=9)

ax.set_xlabel("Prompt Size (tokens)", fontsize=11)
ax.set_ylabel("Mean Latency (seconds)", fontsize=11)
ax.set_title("RQ3: Prompt Size vs Latency — SA5 architecture, o4-mini",
             fontsize=12, fontweight="bold")
ax.legend(loc="lower right")
ax.spines[["top","right"]].set_visible(False)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("rq3_results/fig1_latency_vs_tokens.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("Saved: rq3_results/fig1_latency_vs_tokens.png")

# ── FIG 2: Quality vs prompt size ───────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))

colors = ["#1D9E75" if p>=70 else ("#E6A817" if p>=40 else "#C0392B") 
          for p in summary["pass_rate"]]

bars1 = ax1.bar(summary["size"], summary["qual_mean"], color=colors, edgecolor="white", width=0.6)
for bar, v in zip(bars1, summary["qual_mean"]):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f"{v:.1f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
ax1.axhline(5, color="red", linestyle="--", alpha=0.5, label="Pass threshold (5/8)")
ax1.set_ylabel("Mean Quality Score (0-8)")
ax1.set_title("Quality Score per Prompt Size", fontweight="bold")
ax1.set_ylim(0, 9); ax1.legend()
ax1.spines[["top","right"]].set_visible(False); ax1.grid(axis="y",alpha=0.3)
plt.setp(ax1.get_xticklabels(), rotation=20, ha="right")

bars2 = ax2.bar(summary["size"], summary["pass_rate"], color=colors, edgecolor="white", width=0.6)
for bar, v in zip(bars2, summary["pass_rate"]):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f"{v:.0f}%", ha="center", va="bottom", fontweight="bold", fontsize=10)
ax2.axhline(70, color="red", linestyle="--", alpha=0.5, label="Pass threshold (70%)")
ax2.set_ylabel("Pass Rate (%)")
ax2.set_title("Pass Rate per Prompt Size", fontweight="bold")
ax2.set_ylim(0, 115); ax2.legend()
ax2.spines[["top","right"]].set_visible(False); ax2.grid(axis="y",alpha=0.3)
plt.setp(ax2.get_xticklabels(), rotation=20, ha="right")

plt.tight_layout()
plt.savefig("rq3_results/fig2_quality_per_size.png", dpi=150, bbox_inches="tight")
plt.show(); plt.close()
print("Saved: rq3_results/fig2_quality_per_size.png")

# ── Summary table ────────────────────────────────────────────────────
print(f"\n{'='*75}")
print(f"  RQ3 PROMPT SIZE BENCHMARK SUMMARY")
print(f"{'='*75}")
print(f"  {'Size':<12} {'Tokens':>7} {'Rows':>5} {'Latency':>12} {'Quality':>9} {'Pass%':>6}")
print("-"*75)
for _, row in summary.iterrows():
    print(f"  {row['size']:<12} {int(row['tokens']):>7} {int(row['rows']):>5} "
          f"{row['lat_mean']:>5.2f}s ±{row['lat_std']:>4.1f} {row['qual_mean']:>6.2f}/8  {row['pass_rate']:>5.0f}%")
print("="*75)
if len(x) >= 2:
    print(f"\n  Pearson correlation (tokens vs latency): r = {r:.3f}")
    print(f"  Latency added per 1,000 tokens: {slope*1000:.2f} seconds")


       size  tokens  rows  lat_mean   lat_std  qual_mean  pass_rate
0  XS_1year     245    12   16.0834  4.362166        7.4      100.0
1  S_2years     450    24   17.4386  2.493426        7.0      100.0
2  M_3years     656    36   18.6276  5.930400        7.0      100.0
3  L_4years     862    48   18.4000  2.422922        7.0      100.0


/var/folders/zy/jrpsl1v90xx47xwd7fws6np00000gn/T/ipykernel_93981/4213225424.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()


Saved: rq3_results/fig1_latency_vs_tokens.png
Saved: rq3_results/fig2_quality_per_size.png

  RQ3 PROMPT SIZE BENCHMARK SUMMARY
  Size          Tokens  Rows      Latency   Quality  Pass%
---------------------------------------------------------------------------
  XS_1year         245    12 16.08s ± 4.4   7.40/8    100%
  S_2years         450    24 17.44s ± 2.5   7.00/8    100%
  M_3years         656    36 18.63s ± 5.9   7.00/8    100%
  L_4years         862    48 18.40s ± 2.4   7.00/8    100%

  Pearson correlation (tokens vs latency): r = 0.908
  Latency added per 1,000 tokens: 3.96 seconds


/var/folders/zy/jrpsl1v90xx47xwd7fws6np00000gn/T/ipykernel_93981/4213225424.py:95: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); plt.close()
